# 04 — Quantum Benchmark workflow_update

**NON_FINAL_CONFIG / NON_BASELINE_RUN**. Default cell uses exhaustive 20-bit Exact fallback. QAOA remains optional until per-seed timeout is enforced.

In [ ]:
import subprocess
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "CLAUDE.md").exists():
    ROOT = ROOT.parent
BASE = [
    "--config",
    "configs/base.yaml",
    "--profile",
    "configs/profiles/workflow_update.yaml",
    "--override",
    "configs/provisional/workflow_update_downstream.yaml",
]


def run(name, cmd):
    print("$", " ".join(cmd), flush=True)
    t = time.perf_counter()
    p = subprocess.run(cmd, cwd=ROOT, check=False)
    elapsed = time.perf_counter() - t
    print(f"[{name}] exit={p.returncode} elapsed={elapsed:.2f}s")
    if p.returncode:
        raise RuntimeError(f"{name} failed")
    return elapsed

In [ ]:
exact_s = run(
    "quantum-exact-fallback",
    [
        "uv",
        "run",
        "qshield-quantum",
        "workflow",
        *BASE,
        "--exact-only",
        "--no-warm-start",
    ],
)

In [ ]:
import json

p = ROOT / "artifacts/dev/optimization/workflow_benchmark.json"
b = json.loads(p.read_text())
print("actual_solver", b.get("actual_solver"), "NON_FINAL", b.get("NON_FINAL_CONFIG"))
print("timings", json.dumps(b.get("stage_timings_seconds"), indent=2))